In [59]:
import torch 
import numpy as np 
import h5py
import os, sys
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer, seed_everything
from corpus.swc_mono_test import SWCMonoTestSetH5Dataset
sys.path.append(str(Path.cwd().parent))
sys.path.append('phaselocknet_torch')
from phaselocknet_torch import phaselocknet_model
from phaselocknet_torch import util
import json 
import src.audio_transforms as at
import src.spatial_attn_architecture as arch 



importlib.reload(phaselocknet_model)
importlib.reload(util)

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

%matplotlib inline
import matplotlib.pyplot as plt
from torchaudio.transforms import Resample

In [2]:
dir_model = "/om2/user/msaddler/auditoryutil/models/singletask/batchnorm_arch0_0000_taskW/"


# Option 1: Manually construct the model from config files:
with open(os.path.join(dir_model, "config.json"), "r") as f:
    config_model = json.load(f)
with open(os.path.join(dir_model, "arch.json"), "r") as f:
    architecture = json.load(f)
model = phaselocknet_model.Model(
    config_model=config_model,
    architecture=architecture,
    input_shape=[2, 125_000, 2],  # <-- [batch, timesteps @ 50 kHz sampling rate, channels==2] for sound_localization
    config_random_slice={"size": [50, 20000], "buffer": [0, 500]},
)


# # Option 2: Use the `phaselocknet_model.get_model` function
# model, config_model = phaselocknet_model.get_model(
#     dir_model=dir_model,
#     fn_config="config.json",
#     fn_arch="arch.json",
# )

# Load model weights from torch checkpoint
util.load_model_checkpoint(
    model=model.perceptual_model,
    dir_model=dir_model,
    fn_ckpt="ckpt_BEST.pt",
    weights_only=True,
)

model = model.cuda().eval()
model


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


[load_model_checkpoint] /om2/user/msaddler/auditoryutil/models/singletask/batchnorm_arch0_0000_taskW/ckpt_BEST.pt


Model(
  (peripheral_model): PeripheralModel(
    (body): Sequential(
      (cochlear_filterbank): GammatoneFilterbank(
        (fb): FIRFilterbank()
      )
      (ihc_transduction): IHCTransduction()
      (ihc_lowpass_filter): IHCLowpassFilter()
      (anf_rate_level): SigmoidRateLevelFunction(
        (envelope_function): HilbertEnvelope(
          (hilbert): Hilbert()
        )
      )
      (anf_spike_generator): BinomialSpikeGenerator()
    )
    (head): RandomSlice(
      (crop): RandomCrop(size=(50, 20000), padding=None)
    )
  )
  (perceptual_model): PerceptualModel(
    (body): Sequential(
      (block0_conv): CustomPaddedConv2d(6, 32, kernel_size=(2, 42), stride=(1, 1))
      (block0_relu): ReLU()
      (block0_pool): HanningPooling()
      (block0_norm): SyncBatchNorm(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (block1_conv): CustomPaddedConv2d(32, 64, kernel_size=(2, 18), stride=(1, 1))
      (block1_relu): ReLU()
      (block1_pool): Hannin

In [61]:
stim_path = "/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5"
sr = 50_000
label_type = "CV"
condition = '1-talker-english-different'
dataset = SWCMonoTestSetH5Dataset(h5_path=stim_path,
                                    eval_distractor_cond=condition,
                                    model_sr=sr,
                                    label_type=label_type)


In [62]:
cv_word_map = dataset.get_class_map()

In [63]:
ix_to_word = {v:k for k,v in cv_word_map.items()}

In [64]:
cue, target, distractor, word_int, dist_word_int = dataset[20]


In [65]:
ipd.Audio(target, rate=sr)

In [66]:
snr = 0 
audio_transforms = at.AudioCompose([
                        at.AudioToTensor(),
                        at.Resample(orig_freq=44_100, new_freq=config_model['kwargs_cochlea']['sr_input']),
                        at.CombineWithRandomDBSNR(low_snr=snr, high_snr=snr), 
                        at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                        at.DuplicateChannel(num_output_channels=2),
                        # at.UnsqueezeAudio(dim=0),
# 
                ])

In [67]:
def collate_fn(batch):
    cues = []
    mixtures = []
    labels = []
    for cue, target, distractor, tgt_label, dist_label in batch:
        cue, _ = audio_transforms(cue, None)
        mixture, _ = audio_transforms(target, distractor)
        mixture = mixture.T.reshape(1,-1, 2)
        cue = cue.T.reshape(1,-1, 2)
        cues.append(cue)
        mixtures.append(mixture)
        labels.append(tgt_label)
    cues = torch.cat(cues, dim=0)
    mixtures = torch.cat(mixtures, dim=0)
    labels = torch.tensor(labels)
    return cues, mixtures, labels

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=8,
    num_workers=0,
    shuffle=False,
    collate_fn=collate_fn,
)

cue, mixture, labels  = next(iter(dataloader))

{'label_word_int': Sequential(
   (fc_output): Linear(in_features=512, out_features=801, bias=True)
 )}

In [90]:
importlib.reload(arch)
dir_model = "/om2/user/msaddler/auditoryutil/models/singletask/batchnorm_arch0_0000_taskW/"

# Load the model
model = arch.SaddlerBackBoneLearnedGains(dir_model).cuda()

logits = model(cue.cuda(), mixture.cuda())

[load_model_checkpoint] /om2/user/msaddler/auditoryutil/models/singletask/batchnorm_arch0_0000_taskW/ckpt_BEST.pt


In [94]:
logits['label_word_int'].softmax(-1).argmax(-1).cpu().numpy()

array([  1,   1, 698, 219, 223, 268, 739,   5])

In [103]:
[name for name,param in model.named_parameters() if param.requires_grad]

['gains.attn0.bias',
 'gains.attn0.slope',
 'gains.attn0.threshold',
 'gains.attn1.bias',
 'gains.attn1.slope',
 'gains.attn1.threshold',
 'gains.attn2.bias',
 'gains.attn2.slope',
 'gains.attn2.threshold',
 'gains.attn3.bias',
 'gains.attn3.slope',
 'gains.attn3.threshold',
 'gains.attn4.bias',
 'gains.attn4.slope',
 'gains.attn4.threshold',
 'gains.attn5.bias',
 'gains.attn5.slope',
 'gains.attn5.threshold',
 'gains.attn6.bias',
 'gains.attn6.slope',
 'gains.attn6.threshold']

In [83]:
ipd.Audio(target_eg, rate=sr)

In [84]:
target_reshaped = target_eg.T.reshape(1,-1,2).cuda()

In [85]:
ipd.Audio(target_reshaped.squeeze().T.cpu(), rate=sr)

In [86]:
target_reshaped.shape

torch.Size([1, 125000, 2])

In [87]:
word_int

16

In [ ]:
import pandas as pd
saddler_word_dict = pd.read_csv('/om2/user/msaddler/phaselocknet/data/misc/word_int_encoding.csv')
saddler_word_dict = saddler_word_dict[~saddler_word_dict.cv.isna()].reset_index(drop=True)
saddler_word_dict = {word:int(int_ix) for word, int_ix in zip(saddler_word_dict.word, saddler_word_dict.cv)}
saddler_int_to_word = {int_ix:word for word, int_ix in saddler_word_dict.items()}

: 

In [105]:
model_out = model(mixture.cuda())['label_word_int'].softmax(-1).argmax(-1).cpu().numpy()

In [106]:
[saddler_int_to_word[out] for out in model_out]

['about', 'about', 'their', 'above', 'cases', 'found', 'takes', 'across']

In [107]:
[ix_to_word[label] for label in labels.numpy()]

['about', 'about', 'above', 'above', 'access', 'access', 'across', 'across']

In [29]:
all([saddler_int_to_word[v+1] == word for word, v in cv_word_map.items()])


True

In [117]:
### Dev with torch module 
import yaml
import src.saddler_w_gains_lightning as  lm
import src.spatial_attn_architecture as arch
from pytorch_lightning import seed_everything, Trainer
importlib.reload(lm)
importlib.reload(arch)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)

config_path = "config/binaural_attn/word_task_v10_saddler_backbone_learned_gains.yaml"
config = yaml.load(open(config_path, "r"), Loader=yaml.FullLoader)


model = lm.SaddlerBackBoneLearnedGains(config)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


TypeError: super(type, obj): obj must be an instance or subtype of type

In [118]:
config

{'corpus': {'name': 'spatialized_commonvoice_audioset_scenes',
  'cue_type': 'mixed',
  'task': 'word',
  'root': '/om/scratch/Fri/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v10',
  'mixture_percentages': {'voice_only': 0.5, 'voice_and_location': 0.5},
  'gender_balanced_4M': True,
  'cue_free_percentage': 0.1,
  'v06': True},
 'audio': {'rep_type': 'cochlea_filt',
  'v2_demean': True,
  'rep_kwargs': {'sr': 44100,
   'env_sr': 10000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'binaural': True,
   'rep_on_gpu': True,
   'center_crop': True,
   'out_dur': 2,
   'impulse_len': 0.25,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'cli